In [ ]:
## importing packages
from enum import auto
from statistics import median
import statistics
from unicodedata import decimal
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics
from pprint import pprint
from sklearn.preprocessing import OneHotEncoder
import seaborn as sns
from hyperopt import hp, fmin, tpe, space_eval, Trials
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

In [ ]:
## Open the file
Perovskites = pd.read_csv(r"C:\Users\Perovskites - machine learning\database_perovskitas_ML_multitarget_hotencoding.txt", 
                                delimiter = "\t", decimal = ",")

## remove columns 
Perovskites = Perovskites.drop(["Total_amount"], axis=1)

## remove rows outliers
Perovskites = Perovskites.drop([37,52,154, 56, 15, 65], axis=0, inplace=False)

#### columns removed based on shap feature importance
Perovskites = Perovskites.drop(['S_II','S_II_amount_mL'], axis=1)

### hot encoding
Perovskites_encoded = pd.get_dummies(Perovskites, columns=['A_source','B1_source', 'B2_source', 'X_source'])
print(Perovskites_encoded)

#logfeatures
Perovskites_encoded['Diameter_nm'] = np.log10(Perovskites_encoded["Diameter_nm"])
Perovskites_encoded['Time_min'] = np.log10(Perovskites_encoded["Time_min"])
Perovskites_encoded['B2_amount_mmol'] = np.log10(Perovskites_encoded["B2_amount_mmol"])
Perovskites_encoded['Temperature_K'] = np.log10(Perovskites_encoded["Temperature_K"])
Perovskites_encoded['Bandgap'] = np.log10(Perovskites_encoded["Bandgap"])
Perovskites_encoded['Tolerance_factor_new'] = np.log10(Perovskites_encoded["Tolerance_factor_new"])

## x and y subsets
x = Perovskites_encoded.drop(['B1_amount_mmol', 'X_amount_mmol', 'Time_min', 'Bandgap', 'Amine_amount_mmol'], axis=1)
y = Perovskites_encoded[[ 'B1_amount_mmol',  'X_amount_mmol', 'Time_min','Bandgap', 'Amine_amount_mmol']]

np.random.seed(100)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=25) # 75% training and 25% test

In [ ]:
'''
## hyperopt
####### to optimize hyperparameters, do the lines below
# Define the search space
space = {
    'bootstrap': hp.choice('bootstrap', [True, False]),
    'max_depth': hp.choice('max_depth', range(1, 1000)),
    'max_features': hp.choice('max_features', [1.0, 'sqrt']),
    'min_samples_leaf': hp.choice('min_samples_leaf', range(2, 10)),
    'min_samples_split': hp.choice('min_samples_split', range(2, 10)),
    'n_estimators': hp.choice('n_estimators', range(100, 1000))
}

cv = KFold(n_splits=5, random_state=25, shuffle=True)
# Define the objective function
def objective(params):
    rf = RandomForestRegressor(**params)
    scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='neg_mean_squared_error')
    rmse = np.sqrt(-scores.mean())  # Negative mean squared error to minimize
    return rmse

# Run the hyperparameter optimization
trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=100, trials=trials)

# Get the best hyperparameters
best_params = space_eval(space, best)

# Create a Random Forest regressor with the best hyperparameters
rf = RandomForestRegressor(**best_params)
scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='neg_mean_squared_error')

rf.fit(X_train, y_train)

# Prediction on test set
y_pred = rf.predict(X_test)


print(best_params)
'''

In [ ]:
 #### after defined the search space
best_params = {
    'bootstrap': False,
    'max_depth': 550,
    'max_features': 'sqrt',
    'min_samples_leaf': 2,
    'min_samples_split': 4,
    'n_estimators': 250,
}


# Create a Random Forest regressor with the best hyperparameters - hyperopt
rf = RandomForestRegressor(**best_params)
rf.fit(X_train, y_train)


# Prediction on test set
y_pred = rf.predict(X_test)
y_pred_train = rf.predict(X_train)

In [ ]:
'''
### GridSearch
#### testing ranges for hyperparameters
param_grid = {
    'bootstrap': [True],
    'max_depth': range(10,500,50),
    'max_features': ['sqrt'],
    'min_samples_leaf': range(1,31,5),
    'min_samples_split': range(1,21,5),
    'n_estimators': range(100,500,50)
}

param_grid = {
    'bootstrap': [True],
    'max_depth': [450],
    'max_features': ['sqrt'],
    'min_samples_leaf': [2],
    'min_samples_split': [3],
    'n_estimators': [200],
}

# Create a based model
rf = RandomForestRegressor()
# Instantiate the grid search model
grid_search = GridSearchCV(estimator = rf, param_grid = param_grid, 
                          cv = 5, n_jobs = -1, verbose = 2)

# Fit the grid search to the data
grid_search.fit(X_train, y_train)
pprint(grid_search.best_params_)


# prediction on test set
y_pred=grid_search.predict(X_test)

## training metrics
y_pred_train=grid_search.best_estimator_.predict(X_train)

'''

In [ ]:
# Training metrics
MAE_train_rf = pd.DataFrame(mean_absolute_error(y_train, y_pred_train, multioutput='raw_values'))
RMSE_train_rf = pd.DataFrame(np.sqrt(mean_squared_error(y_train, y_pred_train, multioutput='raw_values')))
R2_train_rf = pd.DataFrame(r2_score(y_train, y_pred_train, multioutput='raw_values'))
train_metrics_rf = pd.concat([MAE_train_rf, RMSE_train_rf, R2_train_rf], axis='columns')
train_metrics_rf.columns = ['MAE_train', 'RMSE_train', 'R2_train']
print(train_metrics_rf)

# Test metrics
MAE_test_rf = pd.DataFrame(mean_absolute_error(y_test, y_pred, multioutput='raw_values'))
RMSE_test_rf = pd.DataFrame(np.sqrt(mean_squared_error(y_test, y_pred, multioutput='raw_values')))
R2_test_rf = pd.DataFrame(r2_score(y_test, y_pred, multioutput='raw_values'))
test_metrics_rf = pd.concat([MAE_test_rf, RMSE_test_rf, R2_test_rf], axis='columns')
test_metrics_rf.columns = ['MAE_test', 'RMSE_test', 'R2_test']
print(test_metrics_rf)

In [ ]:
# Unite dataframes into a single one and export in excel
df_unificado = pd.concat([train_metrics_rf, test_metrics_rf], axis=1)

df_unificado.to_excel(r'C:\Users\taty_\OneDrive\Área de Trabalho\Doutorado\Escrita artigo ML\Planilhas\train_test_RF_GS.xlsx', index=False)

In [ ]:
######### to predict --  testing perovskite: Cs2AgInCl6

## observe the columns in a vertical list
print("The columns of dataframe are:")
for col in x.columns:
    print(col)

## 1) create dataframe with x new informations
X_new = {'Tolerance_factor_new':[3.79],
    'Temperature_K':[453],
    'A_amount_mmol':[0.71],
    'B2_amount_mmol':[0.5],
    'Carboxylic_acid_amount_mmol':[8.9],
    'S_I_amount_mL':[10],
    'Diameter_nm':[20],
    'A_source_cesium_acetate':[0],
    'A_source_cesium_bromide':[0],
    'A_source_cesium_carbonate':[1],
    'A_source_potassium_carbonate':[0],
    'A_source_rubidium_acetate':[0],
    'A_source_rubidium_carbonate':[0],
    'B1_source_cadmium_II_acetate':[0],
    'B1_source_cadmium_II_chloride':[0],
    'B1_source_copper_II_acetate':[0],
    'B1_source_copper_II_chloride':[0],
    'B1_source_just_one_B':[0],
    'B1_source_manganese_II_acetate':[0],
    'B1_source_manganese_II_chloride':[0],
    'B1_source_silver_acetate':[0],
    'B1_source_silver_bromide':[0],
    'B1_source_silver_nitrate':[1],
    'B1_source_silver_oxide':[0],
    'B1_source_sodium_acetate':[0],
    'B1_source_tin_II_bromide':[0],
    'B2_source_antimony_III_acetate':[0],
    'B2_source_antimony_III_bromide':[0], 
    'B2_source_antimony_III_chloride':[0],
    'B2_source_bismuth_III_acetate':[0],
    'B2_source_bismuth_III_bromide':[0],
    'B2_source_bismuth_III_chloride':[0],
    'B2_source_bismuth_III_iodide':[0],
    'B2_source_bismuth_III_neodecanoate':[0],
    'B2_source_cadmium_II_chloride':[0],
    'B2_source_copper_I_bromide':[0],
    'B2_source_copper_I_chloride':[0],
    'B2_source_copper_I_iodide':[0],
    'B2_source_erbium_III_acetate':[0],
    'B2_source_europium_III_chloride':[0],
    'B2_source_germanium_II_iodide':[0],
    'B2_source_hafnium_IV_acetylacetonate':[0],
    'B2_source_indium_III_acetate':[0],
    'B2_source_indium_III_bromide':[0],
    'B2_source_indium_III_chloride':[1],
    'B2_source_manganese_II_acetate':[0],
    'B2_source_praseodymium_III_chloride':[0],
    'B2_source_tin_II_bromide':[0],
    'B2_source_tin_II_chloride':[0],
    'B2_source_tin_II_ethylhexanoate':[0],
    'B2_source_tin_II_iodide':[0],
    'B2_source_tin_II_oxalate':[0],
    'B2_source_tin_IV_bromide':[0],
    'B2_source_tin_IV_chloride':[0],
    'B2_source_tin_IV_iodide':[0],
    'B2_source_titanium_IV_isopropoxide':[0],
    'B2_source_titanium_di_isopropoxide_bis_acetylacetonate':[0],
    'B2_source_ytterbium_II_iodide':[0],
    'B2_source_zirconium_II_carbonate':[0],
    'X_source_ammonium_bromide':[0],
    'X_source_ammonium_iodide':[0],
    'X_source_benzoyl_bromide':[0],
    'X_source_benzoyl_chloride':[0],
    'X_source_bromooctadecane':[0],
    'X_source_bromotrimethylsilane':[0],
    'X_source_chlorotrimethylsilane':[1],
    'X_source_ethylhexanoyl_chloride':[0],
    'X_source_from_metal_salt':[0],
    'X_source_germanium_IV_chloride':[0],
    'X_source_indium_III_bromide':[0],
    'X_source_indium_III_chloride':[0],
    'X_source_indium_III_iodide':[0],
    'X_source_manganese_chloride':[0],
    'X_source_trimethylsilyl_iodide':[0],
    'X_source_zinc_bromide':[0],
    'X_source_zinc_iodide':[0],
}

X_new = pd.DataFrame(data=X_new)

In [ ]:

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

X_new['B2_amount_mmol'] = np.log10(X_new["B2_amount_mmol"])
X_new['Temperature_K'] = np.log10(X_new["Temperature_K"])
X_new['Tolerance_factor_new'] = np.log10(X_new["Tolerance_factor_new"])
X_new['Diameter_nm'] = np.log10(X_new["Diameter_nm"])

In [ ]:

## 2) predict new y
y_new=rf.predict(X_new)

## create dataframe
y_new = pd.DataFrame(data=y_new)
print(y_new)

## add columns names to dataframe
y_new.columns = [ 'B1_amount_mmol',  'X_amount_mmol', 'Time_min','Bandgap', 'Amine_amount_mmol']
print(y_new)

## remove logs from features
y_new['Time_min'] = np.power(10,(y_new["Time_min"]))
y_new['Bandgap'] = np.power(10,(y_new["Bandgap"]))
X_new['Temperature_K'] = np.power(10,(X_new["Temperature_K"]))
X_new['Tolerance_factor_new'] = np.power(10,(X_new["Tolerance_factor_new"]))
X_new['B2_amount_mmol'] = np.power(10,(X_new["B2_amount_mmol"]))
X_new['Diameter_nm'] = np.power(10,(X_new["Diameter_nm"]))

print(y_new)

In [ ]:

## add dataframes X and Y together
Synthesis = pd.concat([X_new,y_new], axis=1)
Synthesis = Synthesis.loc[:,['Tolerance_factor_new','Temperature_K', 'Time_min', 'A_amount_mmol', 'B1_amount_mmol', 'B2_amount_mmol', 'X_amount_mmol',\
                  'Amine_amount_mmol', 'Carboxylic_acid_amount_mmol', 'S_I_amount_mL', 'Diameter_nm', 'Bandgap' ]]

print(Synthesis)

In [ ]:
## export synthesis to excel table
Synthesis.to_excel(r'C:\Users\Perovskites - machine learning\synthesis.xlsx', index=False)